# Capstone B: Own the Pipeline - IMDB Sentiment

**Part B - How It Works. Capstone.**
*Author: Axel Sirota, Data Trainers LLC.*

You have spent Part B opening the black box: tensors and autograd (B4), word and
sentence embeddings (B5), the PyTorch neural-net machinery (B6), and the MLP-on-
word2vec classifier we built together (B7). This capstone is where you prove it.
You will take a brand-new dataset - IMDB movie-review sentiment - and build a
trained sentiment classifier end to end, by yourself.

## What you will do
1. Load IMDB and carve out a clean train / validation / test split.
2. Turn each review into a single feature vector by AVERAGING its pretrained word
   vectors (the `doc_vector` pattern from B5/B7).
3. Fit a quick `LogisticRegression` baseline - the bar your model must beat.
4. Define your own `MLPClassifier(nn.Module)` and train it with the B6 loop.
5. Evaluate on the held-out test set and BEAT the baseline.
6. (Stretch) Run a pretrained transformer on the same reviews and measure the gap.

## Prerequisites
- B4 tensors + autograd, B5 embeddings (`wv`), B6 `nn.Module` + training loop,
  B7 the embeddings-as-features pattern.

## Session format
- ~60-90 min. Mostly hands-on: this is YOUR pipeline. Short demos on toy data,
  then you rebuild on IMDB.

## Runtime
- Colab CPU is fine (the whole pipeline runs on CPU in a few minutes). A GPU only
  helps the optional stretch transformer cell.

## Section 0. Environment Setup

Install the pinned stack, import everything, set the seed, and pick the device.
This is the same environment you used in B5 and B7, so nothing here should surprise
you. The pins matter: gensim needs `numpy<2` and `scipy<1.13`, and we hold
`transformers` at 4.57.1 (never 5.x).

In [ ]:
# Install the pinned course stack (run this first in Google Colab).
# Pins explained:
#   numpy<2      -> gensim 4.3 is not numpy-2 compatible
#   scipy<1.13   -> gensim 4.3 imports scipy.linalg.triu, removed in scipy 1.13
#   gensim==4.3.3-> brings gensim.downloader and the KeyedVectors API from B5
#   transformers==4.57.1 -> stable; do NOT let it resolve to 5.x (forces numpy 2)
!pip install -q "numpy<2" "scipy<1.13" "gensim==4.3.3" \
    "transformers==4.57.1" "datasets>=2.19,<3" "sentence-transformers==3.4.1" \
    "scikit-learn>=1.3" "matplotlib" "seaborn"

# IMPORTANT (Colab): Colab preinstalls numpy 2.x. After this cell finishes, use
# Runtime > Restart session ONCE, then run from the top. If you skip the restart
# you may hit "ImportError: cannot import name 'triu' from 'scipy.linalg'".
print("Install complete. If on Colab, restart the runtime now, then re-run from the top.")

In [ ]:
# Shared imports.
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
import gensim.downloader as api

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
)

warnings.filterwarnings("ignore")

# Reproducibility: seed everything (the same idiom as B4-B7).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device: a torch.device object (used for tensors and models).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
print("Environment ready.")

## The Scenario

The customer-support-platform team you have worked with since Part A has been
leaning on your zero-shot router. Now leadership wants proof the team can ship a
TRAINED model, not just call pretrained ones. They hand you a new, unfamiliar
corpus - IMDB movie-review sentiment - and the brief is blunt:

> "Build a sentiment classifier from raw text, end to end, and beat a sensible
> baseline. No hand-holding."

Here is your plan, and it is exactly the Part B toolkit:

1. **Features.** Turn each review into ONE vector by averaging its pretrained word
   vectors (`wv` from B5). A review becomes a single 100-dim point in space.
2. **Baseline.** Fit a `LogisticRegression` on those vectors. That accuracy is the
   bar you must beat - if your neural net cannot beat a linear model, it is not
   earning its complexity.
3. **Model.** Build your own `MLPClassifier(nn.Module)` and train it with the loop
   from B6.
4. **Proof.** Evaluate on a held-out test set the model never saw.

Then, honestly, you will run a pretrained transformer on the same reviews and
measure how much head-room is left. That gap is the whole argument for Part C.

**Figure: the end-to-end IMDB capstone pipeline from raw review to beating the baseline.**

```mermaid
graph TD
    A[Raw IMDB review text] --> B[doc_vector average pretrained word vectors]
    B --> C[One 100-dim feature per review]
    C --> D[LogisticRegression baseline accuracy]
    C --> E[MLPClassifier trained with B6 loop]
    D --> F[Compare on held-out test set]
    E --> F
    F --> G[MLP beats the baseline]
    G --> H[Measure gap to pretrained transformer]
```

In [ ]:
# Load IMDB from the HuggingFace Hub.
#   - train: 25,000 reviews, test: 25,000 reviews, both perfectly balanced 50/50.
#   - fields: 'text' (the review string), 'label' (0 = negative, 1 = positive).
imdb = load_dataset("imdb")
print(imdb)

# Peek at one positive review.
sample = imdb["train"][0]
print("\nLabel:", sample["label"], "(0=negative, 1=positive)")
print("Review (first 300 chars):")
print(sample["text"][:300], "...")

In [ ]:
# Full IMDB is 50k reviews. Averaging word vectors over all of it on CPU is doable
# but slow for a class. We carve a balanced, reproducible subsample:
#   - TRAIN_N positives + TRAIN_N negatives from imdb['train']
#   - TEST_N  positives + TEST_N  negatives from imdb['test'] (the held-out set)
# Everything is seeded, so every student gets the same rows.
TRAIN_N = 2000   # per class -> 4000 train+val reviews total
TEST_N  = 1000   # per class -> 2000 test reviews total

def balanced_sample(split, n_per_class, seed=SEED):
    df = split.to_pandas()
    pos = df[df["label"] == 1].sample(n_per_class, random_state=seed)
    neg = df[df["label"] == 0].sample(n_per_class, random_state=seed)
    out = pd.concat([pos, neg]).sample(frac=1.0, random_state=seed)  # shuffle
    return out.reset_index(drop=True)

train_full = balanced_sample(imdb["train"], TRAIN_N)
test_df    = balanced_sample(imdb["test"],  TEST_N)

# Split train_full -> train / validation (80/20), stratified by label.
train_df, val_df = train_test_split(
    train_full, test_size=0.2, random_state=SEED, stratify=train_full["label"]
)

print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")
print("train label balance:\n", train_df["label"].value_counts())

## Step 1. The Features: Pretrained Word Vectors

You will NOT train word vectors from scratch. You already have them. In B5 you
loaded GloVe vectors with gensim:

```python
import gensim.downloader as api
wv = api.load("glove-wiki-gigaword-100")   # 400,000 words, 100 dims each
```

`wv` is a `KeyedVectors`: a lookup from a word string to a 100-dim numpy vector.
Two facts you will use:
- `wv[word]` (or `wv.get_vector(word)`) returns the 100-dim vector for a word.
- `word in wv.key_to_index` is the cheap, correct OOV check - GloVe does not know
  every token, so you must skip words it has never seen.

The first `api.load` downloads ~130 MB the first time and caches it. After that it
is instant.

**Figure: doc_vector turns a review into one 100-dim vector by averaging its known word vectors.**

```mermaid
graph TD
    A[Review string] --> B[Lowercase and split into tokens]
    B --> C{word in wv.key_to_index}
    C -->|known| D[Look up 100-dim word vector]
    C -->|OOV| E[Skip the word]
    D --> F[Collect kept word vectors]
    F --> G[Mean-pool average over all vectors]
    G --> H[One 100-dim document vector]
```

In [ ]:
# Load the same pretrained vectors you used in B5 (100-dim GloVe, 400k words).
# First call downloads ~130 MB and caches it; later calls are instant.
wv = api.load("glove-wiki-gigaword-100")
print("Vocab size:", len(wv.key_to_index))
print("Vector dim:", wv.vector_size)          # 100
print("Vector for 'movie' (first 5 dims):", wv["movie"][:5])
print("Is 'zzqx' known?", "zzqx" in wv.key_to_index)   # False -> OOV guard works

In [ ]:
# DEMO (toy): a document vector is just the average of its known word vectors.
# This is the same mean-pool idea as B5's doc_vector / mean_vector helper.
def doc_vector(text, kv=wv, dim=100):
    # Lowercase + naive whitespace tokenization is enough for a bag-of-words feature.
    tokens = text.lower().split()
    # Keep only words the vectors know about (the OOV guard).
    vecs = [kv[t] for t in tokens if t in kv.key_to_index]
    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)   # no known words -> zero vector
    # Mean-pool: average across all kept word vectors -> one dim-D vector.
    return np.mean(vecs, axis=0).astype(np.float32)

# Show it on two tiny "reviews":
v_pos = doc_vector("an absolutely wonderful and moving film")
v_neg = doc_vector("a boring and terrible waste of time")
print("doc_vector shape:", v_pos.shape)        # (100,)
print("pos[:4]:", v_pos[:4])
print("neg[:4]:", v_neg[:4])
# They differ -> the average already carries sentiment signal.
print("vectors differ:", not np.allclose(v_pos, v_neg))

## Lab 1: Build the Feature Matrices (your turn)

You have `doc_vector` and three pandas frames (`train_df`, `val_df`, `test_df`),
each with a `text` column and a `label` column. Turn each split into a feature
matrix and a label vector.

Steps:
1. Write a helper `build_xy(df)` that returns `(X, y)` where:
   - `X` is a 2-D numpy array of shape `(len(df), 100)` - one `doc_vector` per row.
   - `y` is a 1-D numpy array of the labels.
   Hint: apply the demo's document-vector helper to every value in `df['text']`,
   then stack the results into one array. The label array comes straight from
   `df['label']`.
2. Call it for all three splits to get
   `X_train, y_train`, `X_val, y_val`, `X_test, y_test`.

You are done when the verification block prints the right shapes and dtypes
(float32 features, integer labels).

In [ ]:
# Build feature matrices for all three splits.

# 1. Helper: turn a dataframe into (X, y).
def build_xy(df):
    # Apply the document-vector helper to every review string, then stack the
    # resulting vectors into one 2-D array.
    X = None  # YOUR CODE: stack doc_vector over df['text'] -> shape (len(df), 100)
    # The labels come straight from the dataframe's label column, as a numpy array.
    y = None  # YOUR CODE: df['label'] as a numpy array
    return X, y

# 2. Build all three splits.
X_train, y_train = None, None  # YOUR CODE: build_xy on the training frame
X_val,   y_val   = None, None  # YOUR CODE: build_xy on the validation frame
X_test,  y_test  = None, None  # YOUR CODE: build_xy on the test frame

# Verification (provided).
for name, X, y in [("train", X_train, y_train),
                   ("val", X_val, y_val),
                   ("test", X_test, y_test)]:
    if X is not None and y is not None:
        print(f"{name:5s}  X={X.shape} ({X.dtype})  y={y.shape} ({y.dtype})")
# Expect X=(N, 100) float32 and y=(N,) integer for each split.

In [ ]:
# Safety-net (provided): if you skipped Lab 1, build the feature matrices here so
# the baseline and the rest of the notebook still run. If you DID Lab 1, this is a
# no-op (X_train is already set).
if X_train is None:
    print("Lab 1 not completed -> using the safety-net feature builder so you can continue.")
    def build_xy(df):
        X = np.vstack([doc_vector(t) for t in df["text"]]).astype(np.float32)
        y = df["label"].to_numpy()
        return X, y
    X_train, y_train = build_xy(train_df)
    X_val,   y_val   = build_xy(val_df)
    X_test,  y_test  = build_xy(test_df)
    print(f"Fallback features ready: X_train={X_train.shape}, X_test={X_test.shape}")


## Step 2. The Baseline: Logistic Regression

Before training a neural net, set the bar. A `LogisticRegression` on the SAME
features is the honest baseline: if your MLP cannot beat a linear model on the same
inputs, the extra complexity is not earning its keep. (This is also exactly the
LogisticRegression baseline you measured in the B5 homework - same idea, new data.)

We fit it on `X_train, y_train` and score it on the validation set. That number is
`baseline_acc` - the line in the sand your MLP has to clear.

In [ ]:
# Fit a logistic-regression baseline on the averaged-embedding features.
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)

# Score on validation (held out from fitting).
baseline_val_pred = baseline.predict(X_val)
baseline_acc = accuracy_score(y_val, baseline_val_pred)
print(f"Baseline (LogReg) validation accuracy: {baseline_acc:.4f}")
print("This is the bar your MLP must beat.")

## Step 3. The Model: Your Own MLP

Now the part you built in B6 and B7: a small `nn.Module`. The architecture is
deliberately simple, because the features are already dense and informative:

```python
input (100) -> Linear(100, 64) -> ReLU -> Linear(64, 2) -> logits (2)
```

Two things to keep straight (you saw both in B6):
- The final layer outputs 2 RAW logits, one per class. Do NOT add a softmax in the
  model: `nn.CrossEntropyLoss` applies log-softmax internally. Adding your own
  double-counts it.
- This is a Deep Averaging Network: average the word vectors, then push the average
  through a feed-forward net. Simple, fast, and a real baseline architecture.

Below is a TOY demo of the module on 3 random "documents" so you can see the output
shape before you write your own.

**Figure: the MLPClassifier forward pass from a 100-dim feature to two raw logits.**

```mermaid
graph TD
    A[Input 100-dim document vector] --> B[Linear 100 to 64]
    B --> C[ReLU]
    C --> D[Linear 64 to 2]
    D --> E[Two raw logits neg and pos]
    E --> F[CrossEntropyLoss applies log-softmax]
    E --> G[argmax over dim 1 gives predicted class]
```

In [ ]:
# DEMO (toy): define and run a tiny MLP on 3 fake 100-dim "documents".
class _DemoMLP(nn.Module):
    def __init__(self, in_dim=100, hidden=64, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes),   # raw logits, no softmax
        )

    def forward(self, x):
        return self.net(x)

demo_model = _DemoMLP()
fake_docs = torch.randn(3, 100)             # 3 documents, 100-dim each
logits = demo_model(fake_docs)
print("input shape :", fake_docs.shape)     # (3, 100)
print("logits shape:", logits.shape)        # (3, 2) -> one row per doc, 2 classes
print("logits:\n", logits)
# These are raw scores. argmax over dim=1 would give the predicted class.

### Lab 2: Define Your MLPClassifier

Write your own `MLPClassifier(nn.Module)` with the architecture from the demo:
`Linear(100 -> hidden) -> ReLU -> Linear(hidden -> 2)`. Then instantiate it, move
it to `device`, and confirm a forward pass on a small batch returns `(batch, 2)`
logits.

In [ ]:
# Lab 2: define your own MLP.
class MLPClassifier(nn.Module):
    def __init__(self, in_dim=100, hidden=64, n_classes=2):
        super().__init__()
        # YOUR CODE: build the layers. You need a linear in_dim->hidden, a ReLU,
        # and a linear hidden->n_classes that outputs raw logits (no softmax).
        self.net = None  # YOUR CODE

    def forward(self, x):
        # YOUR CODE: pass x through your layers and return the logits.
        return None  # YOUR CODE

# Instantiate and move to device.
model = None  # YOUR CODE: create an MLPClassifier and send it to `device`

# Verification (provided).
if model is not None:
    probe = torch.randn(4, 100).to(device)
    out = model(probe)
    print("Output shape:", out.shape)   # expect (4, 2)
    assert out.shape == (4, 2), "Forward pass should return (batch, 2) logits."
    print("MLPClassifier looks correct.")

In [ ]:
# Safety-net (provided): if you skipped Lab 2, define and instantiate a working
# MLPClassifier here so the optimizer and training loop still run. If you DID Lab 2,
# this is a no-op (model is already set).
if model is None:
    print("Lab 2 not completed -> using the safety-net MLPClassifier so you can continue.")
    class MLPClassifier(nn.Module):
        def __init__(self, in_dim=100, hidden=64, n_classes=2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.ReLU(),
                nn.Linear(hidden, n_classes),
            )
        def forward(self, x):
            return self.net(x)
    model = MLPClassifier().to(device)
    print("Fallback model ready:", model.__class__.__name__)


## Step 4. Train It

This is the B6 loop, unchanged. Wrap the features and labels in tensors, batch them
with a `DataLoader`, and for each epoch:

1. `model.train()`, then for each batch: `optimizer.zero_grad()` -> forward ->
   `loss = criterion(logits, labels)` -> `loss.backward()` -> `optimizer.step()`.
2. `model.eval()` under `torch.no_grad()` to measure validation accuracy.

Two dtype rules that bite everyone (you saw them in B6):
- Features must be `float32` tensors (the model weights are float32).
- Labels for `CrossEntropyLoss` must be `int64` / `long`, shape `(N,)` - class
  indices, NOT one-hot.

The demo below builds the tensors and DataLoaders for you and shows ONE training
step so the moving parts are visible. In Lab 3 you write the epoch loop.

**Figure: the B6 training loop, the five moves repeated over every batch and epoch.**

```mermaid
graph TD
    A[Start epoch model.train] --> B[optimizer.zero_grad clear gradients]
    B --> C[forward pass logits = model xb]
    C --> D[loss = criterion logits and labels]
    D --> E[loss.backward backprop the B4 engine]
    E --> F[optimizer.step update weights]
    F --> G{more batches}
    G -->|yes| B
    G -->|no| H[model.eval no_grad measure val accuracy]
```

In [ ]:
# Wrap numpy features/labels into tensors with the CORRECT dtypes.
def make_loader(X, y, batch_size=64, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)   # features: float32
    y_t = torch.tensor(y, dtype=torch.long)      # labels: int64 for CrossEntropy
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, batch_size=64, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   batch_size=256, shuffle=False)

# DEMO: one training step, so you can see the five moves before writing the loop.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
model.train()
optimizer.zero_grad()            # 1. clear old gradients
logits = model(xb)              # 2. forward
loss = criterion(logits, yb)    # 3. loss (logits vs class indices)
loss.backward()                 # 4. backprop (the B4 engine)
optimizer.step()                # 5. update weights
print(f"One step done. batch loss = {loss.item():.4f}")

### Lab 3: Train the Model (Tier 3 - own it)

Train `model` until it beats the logistic baseline on the held-out validation set.
This is a capstone: no step list, no scaffolding. Everything you need
(`train_loader`, `val_loader`, `criterion`, `optimizer`, `evaluate`, `EPOCHS`,
`baseline_acc`) is already defined above.

In [ ]:
# Lab 3 (Tier 3): train `model` for several epochs and beat `baseline_acc` on
# validation. Provided below are EPOCHS and an evaluate() helper; the training loop
# is yours to write. Reuse train_loader, val_loader, criterion, optimizer.
EPOCHS = 15

def evaluate(model, loader):
    # Provided: returns accuracy over a loader (no gradients).
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

# Write the training loop here. When it is working, set trained_ok = True so the
# safety-net below knows you completed the lab.
trained_ok = None


In [ ]:
# Safety-net (provided): if you skipped Lab 3, train the model here so the test-set
# evaluation has a TRAINED model to score. If you DID Lab 3, trained_ok is True and
# this is a no-op.
trained_ok = globals().get("trained_ok", None)
if trained_ok is None:
    print("Lab 3 not completed -> running the safety-net training loop so you can continue.")
    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
    fallback_val = evaluate(model, val_loader)
    print(f"Fallback training done. val_acc = {fallback_val:.4f}")


### Lab 4: Prove It on the Test Set (Tier 3 - own it)

Score the trained `model` on the held-out test set and show it beats the baseline.
No step list. Produce, at minimum, `mlp_acc` (test accuracy) and `mlp_f1` (macro-F1)
from the model's predictions on `X_test`, score the same `baseline` on the test set,
and plot a confusion matrix. State the one-sentence result: "My MLP scored X on test,
the baseline scored Y, so the MLP beat it by Z points."

In [ ]:
# Lab 4 (Tier 3): evaluate the trained model on the test set and beat the baseline.
# A test_loader and the baseline's test score are provided; the predictions, metrics,
# and confusion-matrix plot are yours to write.
test_loader = make_loader(X_test, y_test, batch_size=256, shuffle=False)
base_test_acc = accuracy_score(y_test, baseline.predict(X_test))

# Produce these from the model's predictions on the test set (leave them None until
# you compute them; the safety-net below fills them in if you skip the lab):
mlp_preds = None
mlp_acc = None
mlp_f1 = None


In [ ]:
# Safety-net (provided): if you skipped Lab 4, compute the test predictions and
# metrics here so the stretch comparison (trans_acc - mlp_acc) still runs. If you DID
# Lab 4, mlp_acc is already set and this is a no-op.
if mlp_acc is None:
    print("Lab 4 not completed -> using the safety-net evaluator so you can continue.")
    test_loader = make_loader(X_test, y_test, batch_size=256, shuffle=False)
    model.eval()
    _preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            _preds.extend(model(xb).argmax(dim=1).cpu().numpy())
    mlp_preds = np.array(_preds)
    mlp_acc = accuracy_score(y_test, mlp_preds)
    mlp_f1  = f1_score(y_test, mlp_preds, average="macro")
    print(f"Fallback metrics ready: test acc={mlp_acc:.4f}  macro-F1={mlp_f1:.4f}")


### Stretch: How Close Did You Get to a Pretrained Transformer?

Your MLP learns sentiment from AVERAGED static word vectors. A pretrained transformer
reads each review WITH context (word order, negation). Run the A2 sentiment pipeline
on a handful of the SAME test reviews and compare. Honest caveat: this transformer was
fine-tuned on SST-2 (short single sentences), and IMDB reviews are long documents, so
this is an indicative comparison, not a like-for-like benchmark - which is itself a
useful lesson about transfer.

In [ ]:
# Stretch (optional, needs the transformers install from Cell 2).
from transformers import pipeline

# The pretrained sentiment model from A2 (DistilBERT fine-tuned on SST-2, ~91% on
# SST-2 dev). device=0 uses GPU if present, else -1 for CPU (pipeline int convention).
clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if device.type == "cuda" else -1,
)

# Score the first 200 test reviews (truncate to 512 chars to keep it quick).
sample_texts = list(test_df["text"].iloc[:200].str.slice(0, 512))
sample_labels = test_df["label"].iloc[:200].to_numpy()
preds = clf(sample_texts, truncation=True)
trans_preds = np.array([1 if p["label"] == "POSITIVE" else 0 for p in preds])
trans_acc = accuracy_score(sample_labels, trans_preds)

print(f"Pretrained transformer acc on 200 IMDB reviews: {trans_acc:.4f}")
print(f"Your MLP test acc                              : {mlp_acc:.4f}")
print(f"Gap (transformer - MLP)                        : {trans_acc - mlp_acc:+.4f}")
print("\nThat gap is what Part C 'earns' by FINE-TUNING a transformer for the task.")

### Homework (async, deeper): Features Matter More Than the Model

You proved an MLP beats a linear baseline on averaged GloVe. Now test a stronger
HYPOTHESIS: better FEATURES beat a fancier model. Swap the 100-d averaged-GloVe
features for 384-d sentence embeddings from `all-MiniLM-L6-v2` (the `embedder` from
B5), keep the SAME MLP architecture (just widen the input to 384), retrain, and
measure the lift. SBERT reads the whole sentence with context, so even an averaged-
word baseline should be left behind.

Tasks:
1. Build a `SentenceTransformer("all-MiniLM-L6-v2")` (this is B5's `embedder`).
2. Encode `train_df`, `val_df`, `test_df` texts to 384-d feature matrices.
3. Instantiate `MLPClassifier(in_dim=384)`, retrain with the SAME loop, evaluate.
4. Compare 384-d-SBERT test accuracy to your 100-d-GloVe `mlp_acc`. How big is the
   lift from features alone, with the model held constant?

In [ ]:
# Homework starter.
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=str(device))

# 1. Encode each split to 384-d features (truncate long reviews for speed).
def sbert_features(df):
    texts = list(df["text"].str.slice(0, 512))
    # YOUR CODE: encode texts with the embedder -> (len(df), 384) float32 array
    return None  # YOUR CODE

Xs_train = None  # YOUR CODE: sbert_features(train_df)
Xs_val   = None  # YOUR CODE
Xs_test  = None  # YOUR CODE

# 2. Same MLP, wider input.
sbert_model = None  # YOUR CODE: MLPClassifier(in_dim=384).to(device)

# 3. Retrain with the same loop pattern (reuse make_loader / evaluate), then:
#    sbert_acc = accuracy on Xs_test, y_test
# 4. print the lift: sbert_acc - mlp_acc

# Reference solution lives in the solution notebook.

## Wrap-Up: You Own the Pipeline

What you just did, solo:
- Turned raw reviews into averaged-embedding features with `doc_vector` (B5).
- Set an honest baseline with `LogisticRegression`.
- Built and trained your own `MLPClassifier` with the B6 loop (the B4 `.backward()`
  engine under the hood).
- Beat the baseline on a held-out test set and read the confusion matrix.
- Measured, honestly, the gap to a pretrained transformer.

### When is this the RIGHT model in production?
The averaged-embedding MLP is cheap, fast, runs on CPU, and is easy to ship and
explain - a great default when latency and cost matter and the accuracy is "good
enough." A fine-tuned transformer buys you accuracy (it reads word order and
negation) at the cost of GPU, latency, and serving complexity. Knowing which to
reach for is the job.

### The ceiling you just hit
Averaging word vectors is bag-of-words: it throws away order and negation, so
"not good" looks positive. That is the hard ceiling on this whole approach.

### Bridge to Part C
Part C breaks that ceiling. In C9 you stop averaging static vectors and instead
FINE-TUNE a contextual transformer (`distilbert-base-uncased`) - the SAME
`.backward()` engine from B4, now updating a real language model - then drop it into
a Gradio chatbot. You measured the gap; C9 closes it.

**One-line bridge:** "You measured the ceiling of averaged features; C9 fine-tunes
DistilBERT to break it, then serves it as the chatbot."

**Figure: why averaged features hit a ceiling and when Part C fine-tuning is worth it.**

```mermaid
flowchart LR
    A[Averaged word vectors bag of words] --> B{loses word order and negation}
    B -->|not good looks positive| C[Hard ceiling on accuracy]
    C --> D{need higher accuracy}
    D -->|no cheap CPU is enough| E[Ship the averaged-embedding MLP]
    D -->|yes| F[Part C fine-tune DistilBERT]
    F --> G[Contextual model serves the Gradio chatbot]
```